# Study 926 — T+1 ⏱️

**On 28 May 2024 the United States halved its settlement cycle. Did the tape notice?**

US cash equities, ETFs and corporate bonds moved from **T+2 to T+1** settlement on
Tuesday **2024-05-28** (SEC Rule 15c6-1(a) as amended). Europe, the UK and most of Asia
did not move. That leaves a fund like **EFA** — US-listed shares that now settle in one
day, European and Japanese holdings that still settle in two — carrying a genuine
settlement mismatch, while **SPY** carries none.

So we run a **difference-in-difference**: measure the same four daily quantities on the
treated funds and on SPY, and ask whether the *gap* between them changed at the switch.
The windows are symmetric — **524 trading days either side**,
2022-04-26 → 2024-05-24 against 2024-05-28 → 2026-06-30.

*Real-tape numbers below are the frozen headline (`docs/results.md`, return fingerprint
`1dea7df6b22f`, 4,802 daily rows); the live cells run the offline synthetic control
and are labelled as such. Total-return closes (`auto_adjust=True`). As-of 2026-06-30.*


## 1. What actually changed

Before May 2024, when you bought a US share you paid for it two business days
later. After May 2024, one. Nothing about *what* the share is worth changed —
only how quickly money and stock have to arrive.

The interesting case is a fund like **EFA**, which owns European and Japanese
shares but trades in New York. Its own shares now settle in one day; the things
it owns still settle in two. Someone has to bridge that day — with cash, with
borrowed stock, with an FX swap. If that cost were large enough, you might expect
to see it somewhere in the price: a jumpier overnight session, a different split
between what the fund does while New York sleeps and what it does while New York
trades, or a different pattern around month-end when flows are heaviest.

> 🔬 **For the quants** — the four outcomes come from the exact identity
> `(1 + r_overnight)(1 + r_intraday) = (1 + r_close-to-close)`. No model, no
> parameter, no fitting; `auto_adjust=True` scales open and close by the same
> factor, so the identity survives dividend adjustment.

## 2. The headline: nothing moved

The measure that matters is the **overnight share of the day's risk** — of all the
variance a fund produces in 24 hours, how much lands between the closing bell and
the next open. If settlement pressure had migrated into the overnight window, this
is where it would show.

In [1]:
R = dict(efa_share=0.023, efa_share_t=1.03, eem_share=-0.017, eem_share_t=-0.73,
         efa_share_pre=0.531, efa_share_post=0.559,
         eem_share_pre=0.589, eem_share_post=0.578,
         spy_share_pre=0.415, spy_share_post=0.42)
print('overnight share of daily variance, before -> after 28 May 2024')
print('  SPY (control) : %.3f -> %.3f' % (R['spy_share_pre'], R['spy_share_post']))
print('  EFA (treated) : %.3f -> %.3f' % (R['efa_share_pre'], R['efa_share_post']))
print('  EEM (treated) : %.3f -> %.3f' % (R['eem_share_pre'], R['eem_share_post']))
print()
print('difference-in-difference (treated minus SPY, after minus before):')
print('  EFA %+.3f  (t = %+.2f)' % (R['efa_share'], R['efa_share_t']))
print('  EEM %+.3f  (t = %+.2f)  <- opposite sign to EFA' % (R['eem_share'], R['eem_share_t']))

overnight share of daily variance, before -> after 28 May 2024
  SPY (control) : 0.415 -> 0.420
  EFA (treated) : 0.531 -> 0.559
  EEM (treated) : 0.589 -> 0.578

difference-in-difference (treated minus SPY, after minus before):
  EFA +0.023  (t = +1.03)
  EEM -0.017  (t = -0.73)  <- opposite sign to EFA


Two funds that should have been treated the same way move in **opposite
directions**, neither anywhere near significance. Note also the *levels*: EFA and
EEM put ~53%–59% of their daily risk into the overnight window against SPY's ~42% — but that is not a settlement fact. **Their "overnight" contains the entire European and Asian trading session.** That mechanical confound is why this study can only ever ask about the *change*, never about the level.

## 3. The two results that looked real — and why they aren't

Two numbers in the whole battery cleared a *t* of 2. Both are cautionary tales.

**Total volatility (EEM, *t* = +3.39).** Look at where it comes from: SPY's own daily move shrank from 83 to 69 bps while EEM's went from 86 to 91. The 'effect' is the US mega-cap tape calming down after 2024 — a fact about SPY, tagged onto EEM by the subtraction.

**Overnight volatility (IWM, *t* = +2.24).** IWM is US small caps: its shares *and* its holdings both moved to T+1, so it has no mismatch to feel. It is the placebo, and the placebo fired.

## 4. The test that settles it

Pretend the settlement change happened on some other date — every quarter across the
sample — and run exactly the same analysis. If 28 May 2024 is special, its *t* should
stand out. Here is where it actually ranks:

In [2]:
placebo = (('EFA', 'r_on', 0.02, 1.48, 3.71, 88, 88), ('EFA', 'abs_on', 0.35, 2.77, 10.77, 82, 88), ('EFA', 'on_var_share', 1.03, 1.94, 8.48, 69, 88), ('EFA', 'abs_cc', 2.18, 4.97, 10.64, 69, 88), ('EEM', 'r_on', 1.11, 2.21, 5.05, 47, 81), ('EEM', 'abs_on', 1.87, 3.45, 5.21, 59, 81), ('EEM', 'on_var_share', 0.73, 2.78, 6.22, 66, 81), ('EEM', 'abs_cc', 3.39, 6.26, 10.08, 61, 81), ('IWM', 'r_on', 0.25, 1.82, 4.25, 89, 93), ('IWM', 'abs_on', 2.24, 2.63, 10.44, 55, 93), ('IWM', 'on_var_share', 0.78, 1.92, 6.0, 72, 93), ('IWM', 'abs_cc', 0.63, 2.05, 7.14, 76, 93))
print('leg  outcome        real |t|   median fake |t|   fake dates that beat it')
for leg, oc, real, med, mx, exc, n in placebo:
    print('%-4s %-13s %7.2f %15.2f %18s' % (leg, oc, real, med, '%d / %d' % (exc, n)))

leg  outcome        real |t|   median fake |t|   fake dates that beat it
EFA  r_on             0.02            1.48            88 / 88
EFA  abs_on           0.35            2.77            82 / 88
EFA  on_var_share     1.03            1.94            69 / 88
EFA  abs_cc           2.18            4.97            69 / 88
EEM  r_on             1.11            2.21            47 / 81
EEM  abs_on           1.87            3.45            59 / 81
EEM  on_var_share     0.73            2.78            66 / 81
EEM  abs_cc           3.39            6.26            61 / 81
IWM  r_on             0.25            1.82            89 / 93
IWM  abs_on           2.24            2.63            55 / 93
IWM  on_var_share     0.78            1.92            72 / 93
IWM  abs_cc           0.63            2.05            76 / 93


On **every** outcome and **every** fund, the *typical* made-up date produces a bigger
difference-in-difference than the real one does. The EEM total-volatility result — the
study's largest *t* — is beaten by 61 of 81 arbitrary dates whose median is 6.26.

The reason is not subtle: 2022–2026 contains a rate-hiking cycle, a rate-cutting cycle,
a tariff shock and a volatility regime change. Cut that tape anywhere and you will find
a 'structural break'. **28 May 2024 is one of the least remarkable dates you could
have picked.**

## 5. Could you trade it anyway?

The one tradable version: buy EFA (or EEM), sell SPY, hold the spread across the
turn of the month — the window where settlement and rebalancing flows are heaviest —
with a one-day execution lag, 3 bps a side and 50 bps/yr of borrow on the short leg.
The lag is not cosmetic: because the calendar signal fires *on* the month-end
session, the days actually held are the **first four trading days of each month**,
and the month-end session itself never makes it into the book.

In [3]:
R = dict(efa_net=(-3.42, 3.68), eem_net=(9.05, 16.11), efa_did=7.11, efa_did_t=0.57,
         eem_did=7.06, eem_did_t=0.43, eem_post_t=1.4, eem_sharpe=0.96, on_days=100)
print('long EFA / short SPY : %+.2f -> %+.2f bps per on-day (net)'
      % R['efa_net'])
print('long EEM / short SPY : %+.2f -> %+.2f bps per on-day (net)'
      % R['eem_net'])
print()
print('change at the switch: EFA %+.2f bps (t %+.2f) | EEM %+.2f bps (t %+.2f)'
      % (R['efa_did'], R['efa_did_t'], R['eem_did'], R['eem_did_t']))
print('EEM post-period alone: Sharpe %+.2f but t = %+.2f on only %d on-days'
      % (R['eem_sharpe'], R['eem_post_t'], R['on_days']))

long EFA / short SPY : -3.42 -> +3.68 bps per on-day (net)
long EEM / short SPY : +9.05 -> +16.11 bps per on-day (net)

change at the switch: EFA +7.11 bps (t +0.57) | EEM +7.06 bps (t +0.43)
EEM post-period alone: Sharpe +0.96 but t = +1.40 on only 100 on-days


The EEM spread's post-period Sharpe of +0.96 is the prettiest number in the study, and
it is not a T+1 effect: it was paying +10.75 bps per on-day *before* the switch too.
The **change** — the only thing this study is testing — is +7 bps with a *t* of +0.43,
and stays there at every cost and borrow assumption we swept.

> 🔬 **For the quants** — a constant per-trade cost cancels in a difference-in-
> difference by construction, which is why the DiD is flat across the whole cost ×
> borrow surface. That invariance is reported deliberately rather than hidden.

## 6. Live check — the machinery works (offline synthetic)

The cells below build a **synthetic** two-fund world in which we *plant* a settlement
break — the treated fund's overnight drift and volatility jump on the switch date —
and a second world where nothing happens. The estimator must find the first and stay
silent on the second. Nothing here touches the real tape.

In [4]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from t_plus_one import data, strategy as st
pl_panel, pl_truth = data.synthetic_panel(signal_strength=1.0, seed=926)
nl_panel, nl_truth = data.synthetic_panel(signal_strength=0.0, seed=926)
pl = st.synthetic_detect(pl_panel, pl_truth)
nl = st.synthetic_detect(nl_panel, nl_truth)
print('SYNTHETIC (not the real tape)')
print('planted break: overnight drift DiD %+.2f bps (t %+.2f) -- must fire'
      % (pl['did_r_on_bps'], pl['t_r_on']))
print('null world   : overnight drift DiD %+.2f bps (t %+.2f) -- must stay quiet'
      % (nl['did_r_on_bps'], nl['t_r_on']))

SYNTHETIC (not the real tape)
planted break: overnight drift DiD +16.47 bps (t +2.73) -- must fire
null world   : overnight drift DiD +1.05 bps (t +0.19) -- must stay quiet


## Verdict

- **Signal — None.** The overnight risk split does not move: +0.023 (*t* = +1.03) for EFA, -0.017 (*t* = -0.73) for EEM — opposite signs, both insignificant, both bootstrap CIs across zero. The two significant results are a total-vol measure driven by SPY's own compression and an overnight-vol move in the domestic **placebo**. And the placebo-date distribution buries everything: the median made-up date beats the real one on every outcome.
- **Tradability — Mirage.** The month-end spread changes by ~+7 bps per on-day with *t* between +0.4 and +0.6, gross, at every cost and borrow assumption. There is nothing to size.
- **The honest reading.** A settlement cycle is plumbing. It changed who funds what overnight — FX swaps, stock loan recalls, ETF creation baskets — none of which appear in a price file. A daily open/close tape was never going to see it, and it didn't.